# PCA Dimensionality Reduction and Animated K-Means Clustering

This self-contained notebook demonstrates:
1. **Extracting and Profiling** a real 4-dimensional dataset.
2. **Dimensionality Reduction (PCA)** to project data down to a visualizable 2D space.
3. **Step-by-step K-Means Clustering** applied with custom logic to trace centroids and point assignments.
4. **Animation Generation** to produce a high-quality GIF of the clustering convergence.

### Dataset Details
- **Source**: UCI Machine Learning Repository via `sklearn.datasets` (Fisher's Iris dataset, 1936).
- **Reason for Selection**: It is a public, real, strictly non-synthetic dataset containing exactly **4 numerical features**, offering an ideal, recognizable baseline for both dimensionality reduction and clustering demonstrations.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Minimal Exploratory Data Analysis (EDA)

We load the Iris dataset and inspect its structure. It features four distinct numerical attributes: sepal length, sepal width, petal length, and petal width.

In [2]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)

print(f"Dataset Shape: {df.shape} (Rows: {df.shape[0]}, Features: {df.shape[1]})")
print(f"Numerical Features: {list(df.columns)}")

display(df.head())

Dataset Shape: (150, 4) (Rows: 150, Features: 4)
Numerical Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


## 2. Dimensionality Reduction via PCA

Before clustering and building the 2D animation, the 4D data is centered and scaled (Z-score normalization). We then apply Principal Component Analysis (PCA) to derive the two most explanatory principal components.

In [3]:
# Standardize the features - critical for PCA to behave symmetrically
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df.values)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print(f"Original Shape: {X_scaled.shape} -> PCA Output Shape: {X_pca.shape}")
print(f"Explained Variance Ratio (PC1, PC2): {pca.explained_variance_ratio_}")
print(f"Total Variance Captured: {pca.explained_variance_ratio_.sum()*100:.2f}%")

Original Shape: (150, 4) -> PCA Output Shape: (150, 2)
Explained Variance Ratio (PC1, PC2): [0.72962445 0.22850762]
Total Variance Captured: 95.81%


## 3. Step-by-Step K-Means Implementation

To build our iteration transparency, we wrap standard K-Means assignments using custom iterative logic. This explicitly computes coordinates so that we track centroid locations and label assignments **every single step** until algorithmic convergence.

In [4]:
k = 3  # Based on standard Iris classes

# 1. Initialize centroids randomly from existing dataset points
initial_indices = np.random.choice(X_pca.shape[0], k, replace=False)
centroids = X_pca[initial_indices]

history_centroids = [centroids]
history_labels = []

converged = False
iteration = 0

while not converged and iteration < 50:
    # E-step: Assign nodes to nearest cluster center using Euclidean distance
    distances = np.linalg.norm(X_pca[:, np.newaxis] - centroids, axis=2)
    labels = np.argmin(distances, axis=1)
    history_labels.append(labels)
    
    # M-step: Calculate new cluster centers by computing the mean cluster assignments
    new_centroids = np.array([X_pca[labels == i].mean(axis=0) if np.sum(labels == i) > 0 
                              else centroids[i] for i in range(k)])
    history_centroids.append(new_centroids)
    
    # Convergence check: Did centroids shift substantially?
    if np.allclose(centroids, new_centroids, atol=1e-5):
        converged = True
    
    centroids = new_centroids
    iteration += 1

print(f"K-Means converged after {iteration} iterations.")
print(f"Tracked {len(history_labels)} state updates for animation.")

K-Means converged after 9 iterations.
Tracked 9 state updates for animation.


## 4. Animating Iterations (GIF Output)

Using `matplotlib.animation.FuncAnimation`, we weave together our stored coordinates and labels into an animation showing point reassignment, centroid paths, and iterative shifts. The output is anchored in the repository root as `kmeans_animation.gif`.

In [5]:
fig, ax = plt.subplots(figsize=(8, 6), dpi=120)
ax.set_title("K-Means Clustering Iterations", fontsize=16, fontweight='bold', pad=20)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

scatter_points = ax.scatter(X_pca[:, 0], X_pca[:, 1], c='gray', s=40, alpha=0.6, edgecolors='k')
scatter_cents = ax.scatter([], [], c='red', marker='X', s=200, edgecolors='white', linewidths=2, zorder=5)

title_text = ax.text(0.5, 0.98, '', transform=ax.transAxes, ha='center', fontsize=12, 
                     bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

ax.set_xlabel('Principal Component 1', fontsize=12)
ax.set_ylabel('Principal Component 2', fontsize=12)
ax.grid(True, linestyle='--', alpha=0.6)

ax.set_xlim(X_pca[:, 0].min() - 0.5, X_pca[:, 0].max() + 0.5)
ax.set_ylim(X_pca[:, 1].min() - 0.5, X_pca[:, 1].max() + 0.5)

arrows = []

def init():
    scatter_cents.set_offsets(np.empty((0, 2)))
    title_text.set_text('')
    return scatter_points, scatter_cents, title_text

def animate(i):
    frame_idx = min(i, len(history_labels) - 1)
    
    # Update 1: Point colors via corresponding cluster assignments
    labels = history_labels[frame_idx]
    c_vals = [colors[label] for label in labels]
    scatter_points.set_color(c_vals)
    scatter_points.set_edgecolors('k')
    
    # Update 2: Current Centroid Positions
    curr_cents = history_centroids[frame_idx]
    scatter_cents.set_offsets(curr_cents)
    
    # Update 3: Movement Trajectory (Arrows)
    while arrows:
        arr = arrows.pop()
        arr.remove()
        
    if frame_idx > 0:
        prev_cents = history_centroids[frame_idx - 1]
        for p_cent, c_cent in zip(prev_cents, curr_cents):
            arr = ax.annotate('', xy=c_cent, xytext=p_cent,
                              arrowprops=dict(arrowstyle="->", color='red', lw=2, alpha=0.7))
            arrows.append(arr)
            
    # Title overlay
    title_text.set_text(f"Iteration: {frame_idx + 1} / {len(history_labels)}")
    
    return [scatter_points, scatter_cents, title_text] + arrows

total_frames = len(history_labels) + 2 
anim = FuncAnimation(fig, animate, init_func=init, frames=total_frames, interval=800, blit=True)

gif_path = '../animations/kmeans_animation.gif'
anim.save(gif_path, dpi=120, writer=PillowWriter(fps=1.5))
plt.close(fig)

print(f"Successfully saved interactive iteration artifact exactly to: {os.path.abspath(gif_path)}")

Successfully saved interactive iteration artifact exactly to: d:\Git-Projects\clustering-algorithms\animations\kmeans_animation.gif
